# LuluCare 360 — Module 3 | `test_economist.py`
### Complete Test Suite — 11 Verified Cases + Structural Assertions

**Project:** LuluCare 360 — AI Complaint-Resolution Co-Pilot  
**Module:** Module 3 Tests  
**File maps to:** `backend/tests/test_economist.py`  

---

## Test Strategy
- Every verdict and reader_output is an **explicit mock** — no dependency on Module 1 or 2
- Profiles use **REAL field values** from `customers.csv` (with `_archetype` stripped)
- Tests are grouped: 11 verified cases → supplementary branch coverage → structural assertions

**To run with pytest:** `pytest -q`  
**To run in notebook:** Execute all cells — assertions raise `AssertionError` on failure


## Cell 1 — Setup: Re-run Economist Inline

Since we are in a notebook, we define all economist functions inline (same code as `economist.ipynb`).  
No file import needed — just run Cell 1 of `economist.ipynb` first, then paste here.

In [1]:
# Re-run all economist cells first (or paste them here).
# This cell just verifies the required functions are available.
import os, re, sys, json

# Enum fallback (same as economist.ipynb Cell 1)
try:
    from shared.enums import (
        GENUINE, SUSPICIOUS, LIKELY_ABUSER,
        CONFIRMED, CONTRADICTED, UNVERIFIED,
        ACTION_ACKNOWLEDGE, ACTION_COUPON, ACTION_WALLET_CREDIT,
        ACTION_REFUND, ACTION_ESCALATE, ACTIONS, EMAIL_ACTIONS,
        REFUND_PICKUP, REFUND_KEEP_ITEM, REFUND_NONE, REFUND_TYPES,
        FRUST_LOW, FRUST_MEDIUM, FRUST_HIGH,
        TIER_GOLD, TIER_SILVER, TIER_PLATINUM,
        ISSUE_DAMAGED_DEFECTIVE,
        VALUE_HIGH, VALUE_MEDIUM, VALUE_LOW,
    )
except Exception:
    GENUINE, SUSPICIOUS, LIKELY_ABUSER = 'GENUINE', 'SUSPICIOUS', 'LIKELY_ABUSER'
    CONFIRMED, CONTRADICTED, UNVERIFIED = 'CONFIRMED', 'CONTRADICTED', 'UNVERIFIED'
    ACTION_ACKNOWLEDGE, ACTION_COUPON = 'ACKNOWLEDGE', 'COUPON'
    ACTION_WALLET_CREDIT, ACTION_REFUND, ACTION_ESCALATE = 'WALLET_CREDIT', 'REFUND', 'ESCALATE'
    ACTIONS = (ACTION_ACKNOWLEDGE, ACTION_COUPON, ACTION_WALLET_CREDIT, ACTION_REFUND, ACTION_ESCALATE)
    EMAIL_ACTIONS = (ACTION_COUPON, ACTION_REFUND, ACTION_WALLET_CREDIT)
    REFUND_PICKUP, REFUND_KEEP_ITEM, REFUND_NONE = 'PICKUP', 'KEEP_ITEM', 'NONE'
    REFUND_TYPES = (REFUND_PICKUP, REFUND_KEEP_ITEM, REFUND_NONE)
    FRUST_LOW, FRUST_MEDIUM, FRUST_HIGH = 'Low', 'Medium', 'High'
    TIER_GOLD, TIER_SILVER, TIER_PLATINUM = 'Gold', 'Silver', 'Platinum'
    ISSUE_DAMAGED_DEFECTIVE = 'Damaged_Defective'
    VALUE_HIGH, VALUE_MEDIUM, VALUE_LOW = 'HIGH', 'MEDIUM', 'LOW'

COUPON_STANDARD, COUPON_GENEROUS = 20, 50
WALLET_CREDIT_BY_BAND = {VALUE_LOW: 50, VALUE_MEDIUM: 100, VALUE_HIGH: 200}
NEW_ACCOUNT_MAX_MONTHS = 2
HIGH_RESALE_THRESHOLD  = 2000
ESCALATE_ORDER_VALUE   = 5000
LOW_CONFIDENCE         = 0.5

def value_band(p):
    if p['loyalty_tier'] == TIER_PLATINUM or p['clv_estimate'] >= 40000: return VALUE_HIGH
    if p['loyalty_tier'] in (TIER_GOLD, TIER_SILVER) or p['clv_estimate'] >= 12000: return VALUE_MEDIUM
    return VALUE_LOW

def refund_logistics(p):
    if p['is_perishable_or_hygiene']: return REFUND_KEEP_ITEM
    if p['resale_value'] >= HIGH_RESALE_THRESHOLD: return REFUND_PICKUP
    if p['reverse_logistics_cost'] > p['resale_value']: return REFUND_KEEP_ITEM
    return REFUND_PICKUP

def should_escalate(verdict, reader, p, proposed_refund):
    if proposed_refund and verdict['genuineness'] == SUSPICIOUS and p['order_value'] > ESCALATE_ORDER_VALUE: return True
    if reader['confidence'] < LOW_CONFIDENCE and value_band(p) == VALUE_HIGH: return True
    return False

def _extract_percent(text):
    m = re.search(r'(\d{1,3})\s*%', text)
    return int(m.group(1)) if m else None

def _confirmed_remedy(verdict):
    reason = str(verdict.get('reason', '')).lower()
    pct = _extract_percent(reason)
    mentions_coupon = ('coupon' in reason) or ('discount' in reason) or (pct is not None)
    if mentions_coupon:
        cp = pct if pct in (COUPON_STANDARD, COUPON_GENEROUS) else (COUPON_GENEROUS if (pct or 0) >= 35 else COUPON_STANDARD)
        return ACTION_COUPON, cp, 0
    return ACTION_REFUND, 0, 0

def choose_action(verdict, reader, p):
    g, claim, issue, frust, band = verdict['genuineness'], verdict['claim_status'], reader['issue_type'], reader['frustration'], value_band(p)
    if claim == CONTRADICTED: return ACTION_ACKNOWLEDGE, 0, 0, 'claim CONTRADICTED by our records -> respond from record, no payout'
    if claim == CONFIRMED:
        action, cp, wc = _confirmed_remedy(verdict)
        if g == LIKELY_ABUSER:
            return action, cp, wc, 'CONFIRMED logged promise honoured; genuineness=LIKELY_ABUSER -> ABUSE_FLAG: capped to promised remedy, no extra remediation, normal logistics'
        return action, cp, wc, 'CONFIRMED logged promise -> honour exactly what was logged'
    if g == LIKELY_ABUSER: return ACTION_ACKNOWLEDGE, 0, 0, 'genuineness LIKELY_ABUSER -> acknowledge only, abuse never pays'
    if p['is_first_purchase'] or p['account_age_months'] <= NEW_ACCOUNT_MAX_MONTHS: return ACTION_COUPON, COUPON_STANDARD, 0, 'new/first-purchase account -> generous-but-capped coupon, verify'
    if g == GENUINE and issue == ISSUE_DAMAGED_DEFECTIVE: return ACTION_REFUND, 0, 0, 'GENUINE + Damaged_Defective -> refund (logistics by economics)'
    if g == GENUINE and frust == FRUST_HIGH and band == VALUE_HIGH: return ACTION_COUPON, COUPON_GENEROUS, 0, 'GENUINE + High frustration + HIGH value -> generous coupon'
    if g == GENUINE and frust == FRUST_MEDIUM:
        if band == VALUE_LOW: return ACTION_WALLET_CREDIT, 0, WALLET_CREDIT_BY_BAND[VALUE_LOW], 'GENUINE + Medium + LOW value -> flat wallet credit'
        return ACTION_COUPON, COUPON_STANDARD, 0, 'GENUINE + Medium frustration -> standard coupon'
    return ACTION_ACKNOWLEDGE, 0, 0, 'GENUINE routine / Low frustration -> acknowledge or small gesture'

def decide(verdict, reader, p):
    action, coupon_percent, wallet_credit, note = choose_action(verdict, reader, p)
    refund_type = refund_logistics(p) if action == ACTION_REFUND else REFUND_NONE
    escalate = should_escalate(verdict, reader, p, proposed_refund=(action == ACTION_REFUND))
    if escalate:
        action = ACTION_ESCALATE
        note   = note + ' | OVERRIDE: escalated to human (high stakes + low certainty)'
    email_trigger = action in EMAIL_ACTIONS
    return {'action': str(action), 'refund_type': str(refund_type), 'coupon_percent': int(coupon_percent),
            'wallet_credit': int(wallet_credit), 'escalate': bool(escalate), 'email_trigger': bool(email_trigger), 'reason': str(note)}

print('All economist functions loaded ✅')


All economist functions loaded ✅


## Cell 2 — Test Helpers & Real Customer Profiles

All profiles below are **real values from `customers.csv`** (verified against the actual file).

In [2]:
# Small helper constructors
def reader(issue, frust='Medium', conf=0.9):
    return {'issue_type': issue, 'frustration': frust, 'confidence': conf}

def verdict(g, c, reason=''):
    return {'genuineness': g, 'claim_status': c, 'reason': reason}

# Real customers.csv profiles (only the fields the Economist reads; _archetype stripped)
PROFILES = {
    'C0005': {'loyalty_tier':'Gold',     'clv_estimate':37040,  'order_value':394,
              'is_first_purchase':False, 'account_age_months':27,
              'is_perishable_or_hygiene':True,  'resale_value':0,     'reverse_logistics_cost':108},
    'C0013': {'loyalty_tier':'Silver',   'clv_estimate':9445,   'order_value':53678,
              'is_first_purchase':False, 'account_age_months':60,
              'is_perishable_or_hygiene':False, 'resale_value':34890, 'reverse_logistics_cost':78},
    'C0001': {'loyalty_tier':'Silver',   'clv_estimate':159720, 'order_value':9164,
              'is_first_purchase':False, 'account_age_months':1,
              'is_perishable_or_hygiene':False, 'resale_value':5956,  'reverse_logistics_cost':101},
    'C0004': {'loyalty_tier':'Silver',   'clv_estimate':14467,  'order_value':4204,
              'is_first_purchase':False, 'account_age_months':41,
              'is_perishable_or_hygiene':False, 'resale_value':1891,  'reverse_logistics_cost':291},
    'C0034': {'loyalty_tier':'Bronze',   'clv_estimate':33188,  'order_value':871,
              'is_first_purchase':False, 'account_age_months':33,
              'is_perishable_or_hygiene':True,  'resale_value':0,     'reverse_logistics_cost':68},
    'C0032': {'loyalty_tier':'Silver',   'clv_estimate':57134,  'order_value':410,
              'is_first_purchase':False, 'account_age_months':49,
              'is_perishable_or_hygiene':False, 'resale_value':164,   'reverse_logistics_cost':180},
    'C0016': {'loyalty_tier':'Gold',     'clv_estimate':16176,  'order_value':2293,
              'is_first_purchase':True,  'account_age_months':0,
              'is_perishable_or_hygiene':False, 'resale_value':917,   'reverse_logistics_cost':253},
    'C0020': {'loyalty_tier':'Bronze',   'clv_estimate':4959,   'order_value':224,
              'is_first_purchase':False, 'account_age_months':61,
              'is_perishable_or_hygiene':True,  'resale_value':0,     'reverse_logistics_cost':118},
    'C0059': {'loyalty_tier':'Platinum', 'clv_estimate':114588, 'order_value':24392,
              'is_first_purchase':False, 'account_age_months':1,
              'is_perishable_or_hygiene':False, 'resale_value':14635, 'reverse_logistics_cost':212},
    'C0018': {'loyalty_tier':'Platinum', 'clv_estimate':117030, 'order_value':66383,
              'is_first_purchase':False, 'account_age_months':20,
              'is_perishable_or_hygiene':False, 'resale_value':43148, 'reverse_logistics_cost':121},
    'C0006': {'loyalty_tier':'Silver',   'clv_estimate':25926,  'order_value':445,
              'is_first_purchase':False, 'account_age_months':14,
              'is_perishable_or_hygiene':False, 'resale_value':200,   'reverse_logistics_cost':121},
}

print(f'Profiles loaded: {len(PROFILES)} customers ✅')


Profiles loaded: 11 customers ✅


## Cells 3–13 — The 11 Verified Test Cases

Each cell is one test. It prints PASS or raises AssertionError with a clear message.

| # | Customer | Scenario | Expected Action |
|---|---|---|---|
| 1 | C0005 | Perishable defect | REFUND + KEEP_ITEM |
| 2 | C0013 | Electronics defect | REFUND + PICKUP |
| 3 | C0001 | Serial abuser, furious, HIGH CLV | ACKNOWLEDGE |
| 4 | C0004 | UNVERIFIED claim — must NOT auto-refund | ACKNOWLEDGE |
| 5 | C0034 | Confirmed promise, perishable | REFUND + KEEP_ITEM |
| 6 | C0032 | Economics: freight > resale | REFUND + KEEP_ITEM |
| 7 | C0016 | First purchase, capped coupon | COUPON 20% |
| 8 | C0020 | Low value, Low frustration | ACKNOWLEDGE |
| 9 | C0059 | Escalation valve fires | ESCALATE |
| 10 | C0018 | Calm Platinum defect (trap case) | REFUND + PICKUP |
| 11 | C0006 | Abuser WITH logged promise | REFUND + PICKUP + ABUSE_FLAG |


### Test Case 1 — C0005: Perishable defect -> REFUND + KEEP_ITEM

In [3]:
# TEST CASE 1: Perishable defect -> REFUND + KEEP_ITEM
d = decide(verdict(GENUINE, UNVERIFIED),
           reader(ISSUE_DAMAGED_DEFECTIVE, 'High'),
           PROFILES['C0005'])
print(json.dumps(d, indent=2))
assert d['action'] == ACTION_REFUND, f'FAIL: {d}'
assert d['refund_type'] == REFUND_KEEP_ITEM, 'Perishable must be KEEP_ITEM'
assert d['email_trigger'] is True, f'FAIL: {d}'
assert d['escalate'] is False, f'FAIL: {d}'
print('TEST 1 PASS ✅')


{
  "action": "REFUND",
  "refund_type": "KEEP_ITEM",
  "coupon_percent": 0,
  "wallet_credit": 0,
  "escalate": false,
  "email_trigger": true,
  "reason": "GENUINE + Damaged_Defective -> refund (logistics by economics)"
}
TEST 1 PASS ✅


### Test Case 2 — C0013: Electronics defect -> REFUND + PICKUP

In [4]:
# TEST CASE 2: Electronics defect -> REFUND + PICKUP
d = decide(verdict(GENUINE, UNVERIFIED),
           reader(ISSUE_DAMAGED_DEFECTIVE, 'High'),
           PROFILES['C0013'])
print(json.dumps(d, indent=2))
assert d['action'] == ACTION_REFUND, f'FAIL: {d}'
assert d['refund_type'] == REFUND_PICKUP, 'resale 34890 >= 2000 -> PICKUP'
assert d['email_trigger'] is True, f'FAIL: {d}'
print('TEST 2 PASS ✅')


{
  "action": "REFUND",
  "refund_type": "PICKUP",
  "coupon_percent": 0,
  "wallet_credit": 0,
  "escalate": false,
  "email_trigger": true,
  "reason": "GENUINE + Damaged_Defective -> refund (logistics by economics)"
}
TEST 2 PASS ✅


### Test Case 3 — C0001: Serial abuser, furious, HIGH CLV -> ACKNOWLEDGE (abuse never pays)

In [5]:
# TEST CASE 3: Serial abuser, furious, HIGH CLV -> ACKNOWLEDGE (abuse never pays)
d = decide(verdict(LIKELY_ABUSER, UNVERIFIED, 'ratio 0.833'),
           reader('Delivery', 'High', conf=0.90),
           PROFILES['C0001'])
print(json.dumps(d, indent=2))
assert d['action'] == ACTION_ACKNOWLEDGE, 'Abuser must get ACKNOWLEDGE'
assert d['email_trigger'] is False, f'FAIL: {d}'
assert d['coupon_percent'] == 0 and d['wallet_credit'] == 0, f'FAIL: {d}'
print('TEST 3 PASS ✅')


{
  "action": "ACKNOWLEDGE",
  "refund_type": "NONE",
  "coupon_percent": 0,
  "wallet_credit": 0,
  "escalate": false,
  "email_trigger": false,
  "reason": "genuineness LIKELY_ABUSER -> acknowledge only, abuse never pays"
}
TEST 3 PASS ✅


### Test Case 4 — C0004: UNVERIFIED claim must NOT auto-refund

In [6]:
# TEST CASE 4: UNVERIFIED claim must NOT auto-refund
d = decide(verdict(GENUINE, UNVERIFIED, 'customer says a rep promised a refund'),
           reader('Refund_Return', 'Low'),
           PROFILES['C0004'])
print(json.dumps(d, indent=2))
assert d['action'] != ACTION_REFUND, 'UNVERIFIED alone must never pay'
assert d['action'] == ACTION_ACKNOWLEDGE, f'FAIL: {d}'
assert d['email_trigger'] is False, f'FAIL: {d}'
print('TEST 4 PASS ✅')


{
  "action": "ACKNOWLEDGE",
  "refund_type": "NONE",
  "coupon_percent": 0,
  "wallet_credit": 0,
  "escalate": false,
  "email_trigger": false,
  "reason": "GENUINE routine / Low frustration -> acknowledge or small gesture"
}
TEST 4 PASS ✅


### Test Case 5 — C0034: CONFIRMED promise, perishable -> REFUND + KEEP_ITEM

In [7]:
# TEST CASE 5: CONFIRMED promise, perishable -> REFUND + KEEP_ITEM
d = decide(verdict(GENUINE, CONFIRMED, 'Customer was assured replacement on last contact.'),
           reader('Refund_Return', 'Medium'),
           PROFILES['C0034'])
print(json.dumps(d, indent=2))
assert d['action'] == ACTION_REFUND, 'Honour the logged promise'
assert d['refund_type'] == REFUND_KEEP_ITEM, 'Perishable -> KEEP_ITEM'
assert d['email_trigger'] is True, f'FAIL: {d}'
print('TEST 5 PASS ✅')


{
  "action": "REFUND",
  "refund_type": "KEEP_ITEM",
  "coupon_percent": 0,
  "wallet_credit": 0,
  "escalate": false,
  "email_trigger": true,
  "reason": "CONFIRMED logged promise -> honour exactly what was logged"
}
TEST 5 PASS ✅


### Test Case 6 — C0032: Economics: freight 180 > resale 164 -> REFUND + KEEP_ITEM

In [8]:
# TEST CASE 6: Economics: freight 180 > resale 164 -> REFUND + KEEP_ITEM
d = decide(verdict(GENUINE, UNVERIFIED),
           reader(ISSUE_DAMAGED_DEFECTIVE, 'Medium'),
           PROFILES['C0032'])
print(json.dumps(d, indent=2))
assert d['action'] == ACTION_REFUND, f'FAIL: {d}'
assert d['refund_type'] == REFUND_KEEP_ITEM, 'freight > resale -> KEEP_ITEM'
assert d['email_trigger'] is True, f'FAIL: {d}'
print('TEST 6 PASS ✅')


{
  "action": "REFUND",
  "refund_type": "KEEP_ITEM",
  "coupon_percent": 0,
  "wallet_credit": 0,
  "escalate": false,
  "email_trigger": true,
  "reason": "GENUINE + Damaged_Defective -> refund (logistics by economics)"
}
TEST 6 PASS ✅


### Test Case 7 — C0016: First purchase, capped coupon, no escalate

In [9]:
# TEST CASE 7: First purchase, capped coupon, no escalate
d = decide(verdict(SUSPICIOUS, UNVERIFIED),
           reader('Delivery', 'High', conf=0.85),
           PROFILES['C0016'])
print(json.dumps(d, indent=2))
assert d['action'] == ACTION_COUPON, f'FAIL: {d}'
assert d['coupon_percent'] == 20, 'First purchase cap at 20%'
assert d['escalate'] is False, 'order_value 2293 < 5000'
assert d['email_trigger'] is True, f'FAIL: {d}'
print('TEST 7 PASS ✅')


{
  "action": "COUPON",
  "refund_type": "NONE",
  "coupon_percent": 20,
  "wallet_credit": 0,
  "escalate": false,
  "email_trigger": true,
  "reason": "new/first-purchase account -> generous-but-capped coupon, verify"
}
TEST 7 PASS ✅


### Test Case 8 — C0020: Low value, Low frustration -> ACKNOWLEDGE, no email

In [10]:
# TEST CASE 8: Low value, Low frustration -> ACKNOWLEDGE, no email
d = decide(verdict(GENUINE, UNVERIFIED),
           reader('App_Technical', 'Low'),
           PROFILES['C0020'])
print(json.dumps(d, indent=2))
assert d['action'] == ACTION_ACKNOWLEDGE, f'FAIL: {d}'
assert d['email_trigger'] is False, f'FAIL: {d}'
print('TEST 8 PASS ✅')


{
  "action": "ACKNOWLEDGE",
  "refund_type": "NONE",
  "coupon_percent": 0,
  "wallet_credit": 0,
  "escalate": false,
  "email_trigger": false,
  "reason": "GENUINE routine / Low frustration -> acknowledge or small gesture"
}
TEST 8 PASS ✅


### Test Case 9 — C0059: Escalation valve fires -> ESCALATE, no email

In [11]:
# TEST CASE 9: Escalation valve fires -> ESCALATE, no email
d = decide(verdict(SUSPICIOUS, UNVERIFIED),
           reader(ISSUE_DAMAGED_DEFECTIVE, 'High', conf=0.45),
           PROFILES['C0059'])
print(json.dumps(d, indent=2))
assert d['action'] == ACTION_ESCALATE, f'FAIL: {d}'
assert d['escalate'] is True, f'FAIL: {d}'
assert d['email_trigger'] is False, 'ESCALATE never emails'
# NOTE: C0059 ratio=0.5 exactly; mock as SUSPICIOUS for escalation test (boundary note for Module 2 owner)
print('TEST 9 PASS ✅')


{
  "action": "ESCALATE",
  "refund_type": "NONE",
  "coupon_percent": 20,
  "wallet_credit": 0,
  "escalate": true,
  "email_trigger": false,
  "reason": "new/first-purchase account -> generous-but-capped coupon, verify | OVERRIDE: escalated to human (high stakes + low certainty)"
}
TEST 9 PASS ✅


### Test Case 10 — C0018: Calm Platinum defect — LOW frustration must NOT block REFUND (trap case)

In [12]:
# TEST CASE 10: Calm Platinum defect — LOW frustration must NOT block REFUND (trap case)
d = decide(verdict(GENUINE, UNVERIFIED),
           reader(ISSUE_DAMAGED_DEFECTIVE, 'Low', conf=0.93),
           PROFILES['C0018'])
print(json.dumps(d, indent=2))
assert d['action'] == ACTION_REFUND, 'Calm genuine defect must still get REFUND'
assert d['refund_type'] == REFUND_PICKUP, 'resale 43148 >= 2000'
assert d['email_trigger'] is True, f'FAIL: {d}'
print('TEST 10 PASS ✅')


{
  "action": "REFUND",
  "refund_type": "PICKUP",
  "coupon_percent": 0,
  "wallet_credit": 0,
  "escalate": false,
  "email_trigger": true,
  "reason": "GENUINE + Damaged_Defective -> refund (logistics by economics)"
}
TEST 10 PASS ✅


### Test Case 11 — C0006: Abuser WITH logged promise -> honour, PICKUP, ABUSE_FLAG in reason

In [13]:
# TEST CASE 11: Abuser WITH logged promise -> honour, PICKUP, ABUSE_FLAG in reason
d = decide(verdict(LIKELY_ABUSER, CONFIRMED, 'Customer was assured replacement on last contact.'),
           reader('Refund_Return', 'High'),
           PROFILES['C0006'])
print(json.dumps(d, indent=2))
assert d['action'] == ACTION_REFUND, 'Honour our logged promise even for abuser'
assert d['refund_type'] == REFUND_PICKUP, 'No KEEP_ITEM generosity for abuser'
assert d['email_trigger'] is True, f'FAIL: {d}'
assert 'ABUSE' in d['reason'].upper(), 'Abuse flag must appear in reason'
print('TEST 11 PASS ✅')


{
  "action": "REFUND",
  "refund_type": "PICKUP",
  "coupon_percent": 0,
  "wallet_credit": 0,
  "escalate": false,
  "email_trigger": true,
  "reason": "CONFIRMED logged promise honoured; genuineness=LIKELY_ABUSER -> ABUSE_FLAG: capped to promised remedy, no extra remediation, normal logistics"
}
TEST 11 PASS ✅


## Cell 14 — Supplementary Branch Coverage

Rules 6 & 7 and the CONTRADICTED path are not in the 11 main cases — tested here.

In [14]:
# Rule 6: GENUINE + High frustration + HIGH value -> COUPON 50%
d = decide(verdict(GENUINE, UNVERIFIED), reader('Billing', 'High'), PROFILES['C0018'])
assert d['action'] == ACTION_COUPON, f'Rule 6 failed: {d}'
assert d['coupon_percent'] == 50, f'Rule 6 percent failed: {d}'
print(f'Rule 6 (High+HIGH -> COUPON 50%)   PASS ✅  got: {d["action"]} {d["coupon_percent"]}%')

# Rule 7: GENUINE + Medium + LOW value -> WALLET_CREDIT 50
d = decide(verdict(GENUINE, UNVERIFIED), reader('Billing', 'Medium'), PROFILES['C0020'])
assert d['action'] == ACTION_WALLET_CREDIT, f'Rule 7 WALLET failed: {d}'
assert d['wallet_credit'] == 50, f'Rule 7 amount failed: {d}'
assert d['email_trigger'] is True
print(f'Rule 7 (Medium+LOW -> WALLET_CREDIT 50)  PASS ✅  got: {d["action"]} AED {d["wallet_credit"]}')

# CONTRADICTED: genuine customer, notes contradict claim -> ACKNOWLEDGE, no email
d = decide(verdict(GENUINE, CONTRADICTED, 'no refund per policy'), reader('Refund_Return', 'High'), PROFILES['C0004'])
assert d['action'] == ACTION_ACKNOWLEDGE, f'CONTRADICTED failed: {d}'
assert d['email_trigger'] is False
print(f'CONTRADICTED -> ACKNOWLEDGE          PASS ✅  got: {d["action"]}')


Rule 6 (High+HIGH -> COUPON 50%)   PASS ✅  got: COUPON 50%
Rule 7 (Medium+LOW -> WALLET_CREDIT 50)  PASS ✅  got: WALLET_CREDIT AED 50
CONTRADICTED -> ACKNOWLEDGE          PASS ✅  got: ACKNOWLEDGE


## Cell 15 — Structural Assertions

Verify the **contract shape**, **enum membership**, **email truth table**, and **native Python types** for all mocks.

In [15]:
ALL_MOCKS = [
    (verdict(GENUINE, UNVERIFIED),              reader(ISSUE_DAMAGED_DEFECTIVE,'High'),     PROFILES['C0005']),
    (verdict(LIKELY_ABUSER, UNVERIFIED),        reader('Delivery','High',0.9),             PROFILES['C0001']),
    (verdict(SUSPICIOUS, UNVERIFIED),           reader('Delivery','High',0.85),            PROFILES['C0016']),
    (verdict(GENUINE, CONFIRMED,'replacement'), reader('Refund_Return','Medium'),          PROFILES['C0034']),
    (verdict(SUSPICIOUS, UNVERIFIED),           reader(ISSUE_DAMAGED_DEFECTIVE,'High',0.45),PROFILES['C0059']),
    (verdict(GENUINE, UNVERIFIED),              reader('App_Technical','Low'),             PROFILES['C0020']),
]

EXPECTED_KEYS = {'action','refund_type','coupon_percent','wallet_credit','escalate','email_trigger','reason'}

print('Running structural assertions...')
for i, (v, r, p) in enumerate(ALL_MOCKS):
    d = decide(v, r, p)
    # Exact contract keys
    assert set(d.keys()) == EXPECTED_KEYS, f'Mock {i}: wrong keys {set(d.keys())}'
    # Enum membership
    assert d['action']         in ACTIONS,      f'Mock {i}: bad action {d["action"]}'
    assert d['refund_type']    in REFUND_TYPES,  f'Mock {i}: bad refund_type {d["refund_type"]}'
    assert d['coupon_percent'] in (0,20,50),     f'Mock {i}: bad coupon {d["coupon_percent"]}'
    assert d['wallet_credit']  >= 0,             f'Mock {i}: negative wallet_credit'
    # Email truth table
    expected_email = d['action'] in EMAIL_ACTIONS
    assert d['email_trigger'] == expected_email, f'Mock {i}: email_trigger mismatch for {d["action"]}'
    # Native Python types (not numpy)
    for k, val in d.items():
        assert type(val).__module__ == 'builtins', f'Mock {i}: {k} is not native Python: {type(val)}'
    assert isinstance(d['escalate'],    bool), f'Mock {i}: escalate not bool'
    assert isinstance(d['email_trigger'],bool), f'Mock {i}: email_trigger not bool'
    assert isinstance(d['coupon_percent'],int), f'Mock {i}: coupon_percent not int'

print('All structural assertions PASS ✅')


Running structural assertions...
All structural assertions PASS ✅


## Cell 16 — Unit Tests: `value_band()` and `refund_logistics()`

In [16]:
print('value_band unit tests:')
assert value_band(PROFILES['C0018']) == VALUE_HIGH,   'C0018 Platinum -> HIGH'
assert value_band(PROFILES['C0032']) == VALUE_HIGH,   'C0032 clv 57134 >= 40000 -> HIGH'
assert value_band(PROFILES['C0016']) == VALUE_MEDIUM, 'C0016 Gold -> MEDIUM'
assert value_band(PROFILES['C0020']) == VALUE_LOW,    'C0020 Bronze clv 4959 -> LOW'
print('  All 4 value_band tests PASS ✅')

print('refund_logistics unit tests:')
assert refund_logistics(PROFILES['C0005']) == REFUND_KEEP_ITEM, 'C0005 perishable -> KEEP_ITEM'
assert refund_logistics(PROFILES['C0013']) == REFUND_PICKUP,    'C0013 resale 34890 -> PICKUP'
assert refund_logistics(PROFILES['C0032']) == REFUND_KEEP_ITEM, 'C0032 freight 180>164 -> KEEP_ITEM'
assert refund_logistics(PROFILES['C0006']) == REFUND_PICKUP,    'C0006 resale 200>freight 121 -> PICKUP'
print('  All 4 refund_logistics tests PASS ✅')


value_band unit tests:
  All 4 value_band tests PASS ✅
refund_logistics unit tests:
  All 4 refund_logistics tests PASS ✅


## Cell 17 — Final Summary

In [17]:
print('=' * 60)
print('  LULUCARE 360 MODULE 3 — ALL TESTS COMPLETE')
print('=' * 60)
print('  11 verified case tests         PASS')
print('   3 supplementary branch tests  PASS')
print('   6-mock structural assertions  PASS')
print('   4 value_band unit tests       PASS')
print('   4 refund_logistics unit tests PASS')
print('=' * 60)
print('  Module 3 contract is clean and ready for integration.')


  LULUCARE 360 MODULE 3 — ALL TESTS COMPLETE
  11 verified case tests         PASS
   3 supplementary branch tests  PASS
   6-mock structural assertions  PASS
   4 value_band unit tests       PASS
   4 refund_logistics unit tests PASS
  Module 3 contract is clean and ready for integration.
